# PlantDoc — Label Quality (5-fold out-of-fold) · proof of concept
**SpecialistRoute · protocol v2 · teacher → student, step 1**

Scores every **training** image with a model that never saw it, to find likely label errors before training a student.

1. **Features:** frozen ImageNet ConvNeXt-Large (384 px) → one vector per train and val image. One subprocess per split, checkpointed, retries at a smaller batch on OOM.
2. **5-fold out-of-fold:** per component, StratifiedGroupKFold on train (reviewed look-alike pairs kept in one fold) → logistic regression → out-of-fold probabilities.
3. **cleanlab** → label-quality score (0–1) + flagged images → review CSV (`is_mislabeled (y/n)`) + contact sheets.
4. **Proof of concept on validation:** the same linear probe trained on (A) all labels, (B) quality-weighted labels, (C) flagged removed.
5. **Your review** → precision of the flags per component.

**Never touches the test set.** Independent of `PlantDoc_Main.ipynb` (only reads the final dataset folders).

**Run:** cells top to bottom. Re-run the feature cell if it stops; finished work is skipped.

**Proof criteria:** (1) most flagged Tomato/Classifier images are confirmed mislabeled in your review; (2) B or C beats A on validation log-loss/accuracy. Only then → GPU student run.

In [ ]:
# ── 1. Setup ──────────────────────────────────────────────────────────────────
# Once per environment:   pip install cleanlab scikit-learn opencv-python pillow
import os
os.environ.setdefault("KERAS_BACKEND", "tensorflow")
import json, gc, sys, time, subprocess
import numpy as np
import pandas as pd
import cv2
from PIL import Image, ImageDraw
from sklearn.linear_model import LogisticRegression, LogisticRegressionCV
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import log_loss
try:
    import cleanlab
    print("cleanlab", cleanlab.__version__)
except ImportError:
    print("cleanlab NOT installed — a simpler self-confidence fallback will be used (pip install cleanlab)")
# TensorFlow is NOT imported here: feature extraction runs in a separate process,
# so this notebook never holds GPU memory.


In [ ]:
# ── 2. Configuration — the only cell you normally edit ───────────────────────
CONFIG_BASE = {
    "dataset_root":      "/home/auc/PlantDoc/Datasets/dataset",    # final train/val/test (prepare_final_dataset)
    "output_dir":        "/home/auc/PlantDoc/Experiments",
    "experiment_prefix": "v2",
}

# Same components and class keywords as PlantDoc_Main.ipynb (keyword match on class-folder names)
COMPONENTS = {
    "Classifier":  {"type": "classifier"},
    "Apple":       {"type": "specialist", "keywords": ["Apple"]},
    "Bell_pepper": {"type": "specialist", "keywords": ["Bell_pepper"]},
    "Corn":        {"type": "specialist", "keywords": ["Corn"]},
    "Potato":      {"type": "specialist", "keywords": ["Potato"]},
    "Tomato":      {"type": "specialist", "keywords": ["Tomato"]},
    "Grape":       {"type": "specialist", "keywords": ["Grape"]},
    "Rest":        {"type": "rest", "keywords": ["Blueberry", "Cherry", "Peach", "Raspberry",
                                                 "Soyabean", "Squash", "Strawberry"]},
}
RUN_COMPONENTS = ["Tomato", "Classifier", "Potato", "Corn", "Bell_pepper", "Apple", "Rest", "Grape"]   # order of scoring

LQ = {
    "backbone": "ConvNeXtLarge",          # frozen ImageNet tf.keras.applications ConvNeXt*
    "img_size": 384,
    "n_folds": 5,
    "fold_seed": 42,
    "Cs": [0.001, 0.01, 0.1, 1.0],        # logistic-regression grid (inner 3-fold CV on train)
    "weight_floor": 0.2,                  # student weight = clip(label_quality, floor, 1)
    "feature_batch_size": 16,
    "near_dup_review_csv": os.path.join(os.path.dirname(CONFIG_BASE["dataset_root"]),
                                        "reports", "near_duplicate_review.csv"),
    "out_dir": os.path.join(CONFIG_BASE["output_dir"], f"{CONFIG_BASE['experiment_prefix']}_label_quality"),
}
os.makedirs(LQ["out_dir"], exist_ok=True)
for k in ("dataset_root", "output_dir"):
    print(f"{k:14s}: {CONFIG_BASE[k]}  {'OK' if os.path.isdir(CONFIG_BASE[k]) else 'NOT FOUND'}")
print(f"{'review csv':14s}: {LQ['near_dup_review_csv']}  {'OK' if os.path.exists(LQ['near_dup_review_csv']) else 'NOT FOUND (folds will not be grouped)'}")
print(f"{'outputs':14s}: {LQ['out_dir']}")


In [ ]:
# ── 3. Data helpers (train + val only; test is never read) ────────────────────
TF_IMAGE_EXT = (".bmp", ".gif", ".jpeg", ".jpg", ".png")


def final_split_dirs():
    return {s: os.path.join(CONFIG_BASE["dataset_root"], s) for s in ("train", "val")}


def _class_dirs(root):
    return sorted(d for d in os.listdir(root) if os.path.isdir(os.path.join(root, d)))


def _matches(cls, keywords):
    return any(kw.lower() in cls.lower() for kw in keywords)


def species_of(class_folder):
    for name, p in COMPONENTS.items():
        if p["type"] == "specialist" and _matches(class_folder, p["keywords"]):
            return name
    return "Rest"


def component_class_names(name):
    p = COMPONENTS[name]
    if p["type"] == "classifier":
        return sorted({species_of(c) for c in _class_dirs(final_split_dirs()["train"])})
    return [c for c in _class_dirs(final_split_dirs()["train"]) if _matches(c, p["keywords"])]


def split_table(split):
    d = final_split_dirs()[split]
    return pd.DataFrame([{"image": os.path.join(d, c, f), "true_class": c, "true_species": species_of(c)}
                         for c in _class_dirs(d) for f in sorted(os.listdir(os.path.join(d, c)))
                         if f.lower().endswith(TF_IMAGE_EXT)])


# sanity: every class owned by exactly one component, same classes in train and val
_train_classes = _class_dirs(final_split_dirs()["train"])
assert _train_classes == _class_dirs(final_split_dirs()["val"]), "train/val class folders differ"
for c in _train_classes:
    owners = [n for n, p in COMPONENTS.items() if p["type"] != "classifier" and _matches(c, p["keywords"])]
    assert len(owners) == 1, f"class {c} matched by {owners}"
for s in ("train", "val"):
    t = split_table(s)
    print(f"{s:5s}: {len(t)} images, {t['true_class'].nunique()} classes")
print("Classifier classes:", component_class_names("Classifier"))


In [ ]:
# ── 4. Feature extraction (definitions) — one subprocess per split, checkpointed ──
LQ_JOBS = os.path.join(LQ["out_dir"], "feature_jobs.json")
LQ_WORKER = os.path.join(LQ["out_dir"], "feature_worker.py")

_FEATURE_WORKER_SRC = r'''
import os, sys, json
os.environ.setdefault("KERAS_BACKEND", "tensorflow")
os.environ["TF_FORCE_GPU_ALLOW_GROWTH"] = "true"
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
import numpy as np
import tensorflow as tf
for g in tf.config.list_physical_devices("GPU"):
    try:
        tf.config.experimental.set_memory_growth(g, True)
    except Exception:
        pass


def load_image(path, size):          # identical to _load_tf_image in section 8
    img = tf.image.decode_image(tf.io.read_file(path), channels=3, expand_animations=False)
    return tf.image.resize(img, [size, size], method="bilinear").numpy().astype(np.float32)


def save(path, **arrays):
    tmp = path + ".tmp"
    with open(tmp, "wb") as f:
        np.savez(f, **arrays)
    os.replace(tmp, path)


def main(jobs_file, idx, batch_size, save_every):
    job = json.load(open(jobs_file))[idx]
    paths, out, size = job["paths"], job["out"], job["img_size"]
    partial = out + ".partial.npz"
    feats = []
    if os.path.exists(partial):
        z = np.load(partial, allow_pickle=False)
        if list(z["paths"]) == paths:
            feats = list(z["features"])
            print(f"    resuming from image {len(feats)}/{len(paths)}", flush=True)
    ctor = getattr(tf.keras.applications, job["backbone"])
    model = ctor(include_top=False, weights="imagenet", pooling="avg", input_shape=(size, size, 3))
    n = 0
    for i in range(len(feats), len(paths), batch_size):
        x = np.stack([load_image(p, size) for p in paths[i:i + batch_size]])
        feats.extend(np.asarray(model(x, training=False)))     # ConvNeXt normalises internally
        n += 1
        if n % save_every == 0:
            save(partial, features=np.stack(feats), paths=np.array(paths))
            print(f"    {len(feats)}/{len(paths)}", flush=True)
    save(out, features=np.stack(feats), paths=np.array(paths))
    if os.path.exists(partial):
        os.remove(partial)
    print(f"    saved {os.path.basename(out)} ({len(paths)} images)", flush=True)


if __name__ == "__main__":
    main(sys.argv[1], int(sys.argv[2]), int(sys.argv[3]), int(sys.argv[4]))
'''


def _feature_file(split):
    return os.path.join(LQ["out_dir"], f"features_{split}_{LQ['backbone']}_{LQ['img_size']}.npz")


def _features_ok(split):
    f = _feature_file(split)
    if not os.path.exists(f):
        return False
    try:
        return list(np.load(f, allow_pickle=False)["paths"]) == list(split_table(split)["image"])
    except Exception:
        return False


def extract_features(splits=("train", "val"), batch_size=16, save_every=5, max_failures=3):
    """Frozen-backbone features, one subprocess per split, checkpointed. Re-run until complete."""
    if not LQ["backbone"].lower().startswith("convnext"):
        raise ValueError("Worker assumes a ConvNeXt backbone (built-in normalisation).")
    os.makedirs(LQ["out_dir"], exist_ok=True)
    jobs = [{"split": s, "paths": list(split_table(s)["image"]), "out": _feature_file(s),
             "backbone": LQ["backbone"], "img_size": LQ["img_size"]} for s in splits]
    json.dump(jobs, open(LQ_JOBS, "w"))
    open(LQ_WORKER, "w").write(_FEATURE_WORKER_SRC)
    for idx, job in enumerate(jobs):
        if _features_ok(job["split"]):
            print(f"[Features] {job['split']}: cached")
            continue
        bs, failures = batch_size, 0
        while True:
            print(f"[Features] {job['split']}: {len(job['paths'])} images (batch {bs})", flush=True)
            proc = subprocess.Popen([sys.executable, LQ_WORKER, LQ_JOBS, str(idx), str(bs), str(save_every)],
                                    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
            tail = []
            for line in proc.stdout:
                tail = (tail + [line.rstrip()])[-15:]
                if line.startswith("    ") and not line.startswith("    File"):
                    print(line.rstrip(), flush=True)
            proc.wait()
            if proc.returncode == 0:
                break
            failures += 1
            print(f"    FAILED (exit {proc.returncode}). Last lines:\n" + "\n".join("      " + t for t in tail[-6:]))
            if bs > 1:
                bs = max(1, bs // 2)
            elif failures >= max_failures:
                print("    giving up")
                return False
    return all(_features_ok(s) for s in splits)




In [ ]:
# ── 5. Label-quality scoring + proof of concept (definitions) ─────────────────
def _load_features(split):
    if not _features_ok(split):
        raise RuntimeError(f"Features for {split} missing — run extract_features().")
    return np.load(_feature_file(split), allow_pickle=False)["features"]


def _component_rows(name, split):
    """This component's images in `split` and their given labels (species for the classifier)."""
    df = split_table(split).reset_index(drop=True)
    classes = component_class_names(name)
    if COMPONENTS[name]["type"] == "classifier":
        df["given_label"] = df["true_species"]
    else:
        df = df[df["true_class"].isin(classes)].copy()
        df["given_label"] = df["true_class"]
    df["row"] = df.index
    df["rel"] = df["true_class"] + "/" + df["image"].map(os.path.basename)
    df["y"] = df["given_label"].map({c: i for i, c in enumerate(classes)})
    return df.reset_index(drop=True), classes


def _near_dup_groups(rels):
    """Group ids that keep reviewed look-alike pairs (answered 'n') in the same fold."""
    parent = {r: r for r in rels}

    def find(a):
        while parent[a] != a:
            parent[a] = parent[parent[a]]
            a = parent[a]
        return a

    path, linked = LQ["near_dup_review_csv"], 0
    if os.path.exists(path):
        for r in pd.read_csv(path).itertuples():
            a, b = str(r.a_rel), str(r.b_rel)
            if str(r.is_duplicate).strip().lower() != "y" and a in parent and b in parent:
                parent[find(a)] = find(b)
                linked += 1
    else:
        print(f"  ! {path} not found — folds are not grouped (set LQ['near_dup_review_csv'])")
    roots = {r: find(r) for r in rels}
    ids = {g: i for i, g in enumerate(sorted(set(roots.values())))}
    return np.array([ids[roots[r]] for r in rels]), linked


def _predict_full(clf, X, n_classes):
    """predict_proba with a column for every class (a fold may miss a rare class)."""
    p = np.zeros((len(X), n_classes))
    p[:, clf.classes_] = clf.predict_proba(X)
    return p


def _image_quality(paths):
    rows = []
    for p in paths:
        img = cv2.imread(p)
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        rows.append({"blur_laplacian_var": float(cv2.Laplacian(gray, cv2.CV_64F).var()),
                     "brightness": float(gray.mean()), "contrast": float(gray.std())})
    return pd.DataFrame(rows)


def contact_sheet(df, out_prefix, per_page=40, cols=5, tile=200):
    """Thumbnails of flagged images with given vs predicted label, for the y/n review."""
    pages = []
    for start in range(0, len(df), per_page):
        chunk = df.iloc[start:start + per_page]
        rows = (len(chunk) + cols - 1) // cols
        sheet = Image.new("RGB", (cols * tile, rows * (tile + 46)), "white")
        draw = ImageDraw.Draw(sheet)
        for k, r in enumerate(chunk.itertuples()):
            x, y = (k % cols) * tile, (k // cols) * (tile + 46)
            sheet.paste(Image.open(r.image).convert("RGB").resize((tile, tile)), (x, y))
            draw.text((x + 3, y + tile + 2), f"#{start + k + 1} {os.path.basename(r.image)[:28]}", fill="black")
            draw.text((x + 3, y + tile + 16), f"given: {r.given_label[:30]}", fill="darkred")
            draw.text((x + 3, y + tile + 30), f"pred:  {r.oof_pred[:24]} ({r.label_quality:.2f})", fill="darkblue")
        path = f"{out_prefix}_p{len(pages) + 1}.jpg"
        sheet.save(path, quality=90)
        pages.append(path)
    return pages


def label_quality(name):
    """5-fold OOF label-quality scores for one component's training images."""
    df, classes = _component_rows(name, "train")
    K = len(classes)
    if K < 2:
        print(f"\n[LabelQuality] {name}: only one class — nothing to score, skipped")
        return None
    X = _load_features("train")[df["row"].to_numpy()]
    y = df["y"].to_numpy()
    groups, linked = _near_dup_groups(list(df["rel"]))
    cv = StratifiedGroupKFold(n_splits=LQ["n_folds"], shuffle=True, random_state=LQ["fold_seed"])
    oof = np.zeros((len(df), K))
    fold_of = np.zeros(len(df), dtype=int)
    for f, (tr, te) in enumerate(cv.split(X, y, groups)):
        sc = StandardScaler().fit(X[tr])
        inner = min(3, int(np.bincount(y[tr], minlength=K)[np.bincount(y[tr], minlength=K) > 0].min()))
        C = (LogisticRegressionCV(Cs=LQ["Cs"], cv=inner, max_iter=5000).fit(sc.transform(X[tr]), y[tr]).C_[0]
             if inner >= 2 else 1.0)
        clf = LogisticRegression(C=C, max_iter=5000).fit(sc.transform(X[tr]), y[tr])
        oof[te] = _predict_full(clf, sc.transform(X[te]), K)
        fold_of[te] = f
    oof = np.clip(oof, 1e-6, 1)
    oof /= oof.sum(1, keepdims=True)

    try:
        from cleanlab.rank import get_label_quality_scores
        from cleanlab.filter import find_label_issues
        quality = get_label_quality_scores(y, oof, method="self_confidence")
        issue = np.zeros(len(df), dtype=bool)
        if K > 1:
            issue = find_label_issues(y, oof, filter_by="prune_by_noise_rate")
        method = "cleanlab (confident learning)"
    except ImportError:
        quality = oof[np.arange(len(df)), y]
        thr = np.array([quality[y == k].mean() if (y == k).any() else 1.0 for k in range(K)])
        issue = (oof.argmax(1) != y) & (oof.max(1) >= thr[oof.argmax(1)])
        method = "self-confidence fallback (pip install cleanlab for confident learning)"

    df["fold"] = fold_of
    df["oof_pred"] = np.array(classes)[oof.argmax(1)]
    df["p_given"] = oof[np.arange(len(df)), y]
    df["label_quality"] = quality
    df["label_issue"] = issue
    df["student_weight"] = np.clip(quality, LQ["weight_floor"], 1.0)
    df = pd.concat([df, _image_quality(list(df["image"]))], axis=1)
    tf_csv = os.path.join(CONFIG_BASE["output_dir"], f"{CONFIG_BASE['experiment_prefix']}_teacher", f"{name}_teacher_train.csv")
    if os.path.exists(tf_csv):                       # cross-check with section 12, if it was run
        t = pd.read_csv(tf_csv)[["image", "seeds_disagreeing", "p_given"]].rename(
            columns={"p_given": "teacher_p_given"})
        df = df.merge(t, on="image", how="left")

    acc = (df["oof_pred"] == df["given_label"]).mean()
    print(f"\n{'=' * 70}\n  LABEL QUALITY — {name}: {len(df)} train images, {K} classes, "
          f"{LQ['n_folds']} folds ({linked} look-alike pairs kept together)\n{'=' * 70}")
    print(f"  out-of-fold probe accuracy : {100 * acc:.2f}%   ({method})")
    print(f"  flagged label issues       : {int(issue.sum())} ({100 * issue.mean():.1f}%)")
    per = df.groupby("given_label").agg(images=("image", "size"), oof_acc=("p_given", lambda s: 0.0),
                                        mean_quality=("label_quality", "mean"), flagged=("label_issue", "sum"))
    per["oof_acc"] = df.assign(ok=df["oof_pred"] == df["given_label"]).groupby("given_label")["ok"].mean()
    print(per.round(3).to_string())
    corr = df[["label_quality", "blur_laplacian_var", "brightness", "contrast"]].corr(method="spearman")["label_quality"]
    print("  Spearman(label quality, image quality): " +
          ", ".join(f"{k} {v:+.2f}" for k, v in corr.drop("label_quality").items()))
    if "seeds_disagreeing" in df:
        print(f"  flagged here AND rejected by >=3 teacher seeds: "
              f"{int((df['label_issue'] & (df['seeds_disagreeing'] >= 3)).sum())}")

    out_dir = LQ["out_dir"]
    os.makedirs(out_dir, exist_ok=True)
    df.drop(columns=["row"]).to_csv(os.path.join(out_dir, f"{name}_label_quality.csv"), index=False)
    rev = df[df["label_issue"]].sort_values("label_quality").copy()
    rev["is_mislabeled (y/n)"] = ""
    rev["correct_label_if_known"] = ""
    rev_csv = os.path.join(out_dir, f"{name}_review.csv")
    keep = ["image", "rel", "given_label", "oof_pred", "label_quality", "p_given", "blur_laplacian_var",
            "is_mislabeled (y/n)", "correct_label_if_known"]
    rev[keep].to_csv(rev_csv, index=False)
    sheets = contact_sheet(rev, os.path.join(out_dir, f"{name}_review_sheet")) if len(rev) else []
    print(f"  review: {rev_csv}" + (f" + {len(sheets)} contact sheet(s)" if sheets else ""))
    return df


def probe_effect(name, lq_df=None):
    """Proof of concept on VALIDATION: same linear probe trained on (A) all labels,
    (B) labels weighted by quality, (C) flagged images removed. Higher val accuracy /
    lower val log-loss for B or C = evidence that cleaning the labels helps."""
    if len(component_class_names(name)) < 2:
        return pd.DataFrame()
    lq_df = lq_df if lq_df is not None else pd.read_csv(os.path.join(LQ["out_dir"], f"{name}_label_quality.csv"))
    tr, classes = _component_rows(name, "train")
    va, _ = _component_rows(name, "val")
    tr = tr.merge(lq_df[["image", "student_weight", "label_issue"]], on="image", how="left")
    K = len(classes)
    Xtr, Xva = _load_features("train")[tr["row"]], _load_features("val")[va["row"]]
    sc = StandardScaler().fit(Xtr)
    Xtr, Xva = sc.transform(Xtr), sc.transform(Xva)
    ytr, yva = tr["y"].to_numpy(), va["y"].to_numpy()
    C = LogisticRegressionCV(Cs=LQ["Cs"], cv=3, max_iter=5000).fit(Xtr, ytr).C_[0]   # chosen on TRAIN, shared by all arms
    keep = ~tr["label_issue"].astype(bool).to_numpy()
    arms = {"A all labels": (np.ones(len(tr), bool), None),
            "B weighted": (np.ones(len(tr), bool), tr["student_weight"].to_numpy()),
            "C flagged removed": (keep, None)}
    rows = []
    for arm, (m, w) in arms.items():
        clf = LogisticRegression(C=C, max_iter=5000).fit(Xtr[m], ytr[m], sample_weight=None if w is None else w[m])
        p = np.clip(_predict_full(clf, Xva, K), 1e-7, 1)
        rows.append({"component": name, "arm": arm, "train_images": int(m.sum()),
                     "val_acc": float((p.argmax(1) == yva).mean()),
                     "val_logloss": float(log_loss(yva, p / p.sum(1, keepdims=True), labels=list(range(K)))),
                     "val_correct": int((p.argmax(1) == yva).sum()), "val_n": len(yva)})
    return pd.DataFrame(rows)




def review_summary():
    """After you fill 'is_mislabeled (y/n)' in the review CSVs: precision of the flags."""
    rows = []
    for name in COMPONENTS:
        f = os.path.join(LQ["out_dir"], f"{name}_review.csv")
        if not os.path.exists(f):
            continue
        r = pd.read_csv(f)
        ans = r["is_mislabeled (y/n)"].astype(str).str.strip().str.lower()
        done = ans.isin(["y", "n"])
        rows.append({"component": name, "flagged": len(r), "reviewed": int(done.sum()),
                     "confirmed_y": int((ans == "y").sum()), "rejected_n": int((ans == "n").sum()),
                     "precision": round((ans == "y").sum() / done.sum(), 3) if done.any() else np.nan})
    out = pd.DataFrame(rows)
    print(out.to_string(index=False))
    out.to_csv(os.path.join(LQ["out_dir"], "review_summary.csv"), index=False)
    return out


print("Label-quality scoring defined.")


In [ ]:
# ── 6. RUN: features (GPU, ~10–20 min). Re-run this cell until it prints True ──
features_ready = extract_features(splits=("train", "val"), batch_size=LQ["feature_batch_size"])
print("features ready:", features_ready)


In [ ]:
# ── 7. RUN: 5-fold scoring + proof of concept (CPU, minutes) ───────────────────
assert features_ready, "Run the feature cell until it prints True"
lq_results, poc = {}, []
for name in RUN_COMPONENTS:
    d = label_quality(name)
    if d is not None:
        lq_results[name] = d
        poc.append(probe_effect(name, d))
poc = pd.concat(poc, ignore_index=True)
print("\n" + "=" * 70 + "\n  PROOF OF CONCEPT — linear probe, judged on VALIDATION\n" + "=" * 70)
print(poc.assign(val_acc=(100 * poc["val_acc"]).round(2), val_logloss=poc["val_logloss"].round(4)).to_string(index=False))
poc.to_csv(os.path.join(LQ["out_dir"], "proof_of_concept_val.csv"), index=False)

summary = pd.DataFrame([{"component": n, "train_images": len(d), "flagged": int(d["label_issue"].sum()),
                         "oof_probe_acc": round(100 * (d["oof_pred"] == d["given_label"]).mean(), 2),
                         "mean_label_quality": round(d["label_quality"].mean(), 3)} for n, d in lq_results.items()])
print("\n" + summary.to_string(index=False))
summary.to_csv(os.path.join(LQ["out_dir"], "label_quality_summary.csv"), index=False)
print(f"\nReview: open {LQ['out_dir']}/<component>_review_sheet_p*.jpg and fill <component>_review.csv (y/n)")


## 8. Your review
For each component (start with **Tomato**), open `<component>_review_sheet_p*.jpg` next to `<component>_review.csv` and fill:
- `is_mislabeled (y/n)` — **y** = the label is wrong; **n** = label is right (hard or unclear image is still **n**)
- `correct_label_if_known` — optional

Write your criteria down before starting. Then run the next cell.

In [ ]:
# ── 8. Review summary (re-run any time) ─────────────────────────────────────────
review_table = review_summary()
